# Synthetic Data Evaluation Pipeline

**Research question:** Given any tabular binary-classification dataset, which synthetic
generation method (Bootstrap, GMM, CVAE) best reproduces the real data — and does
feature-space complexity explain why methods succeed or fail?

**Pipeline (universal — same logic runs on every dataset)**
1. Subsample real data to a controlled size
2. Select a feature subset (full / forward top-k / reverse drop-k / drop-one)
3. Train each generator on the subset
4. Evaluate synthetic vs real across three axes:
   - Discriminability (RF AUC / F1)
   - Utility (TSTR F1 vs TRTR baseline)
   - Fidelity (KLD, correlation difference)

**Structure**
1. Imports & configuration
2. Dataset summary
3. Helper functions
4. Experiment runner
5. Run all experiments

---
## 1. Imports & Configuration

In [1]:
import sys
from pathlib import Path

ROOT = Path().resolve().parents[0]
sys.path.append(str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
import matplotlib
matplotlib.use('Agg') 
from loaders import load_breast, load_diabetes, load_HIV
from models.cvae import train_cvae_on_arrays, sample_cvae_dataset
from models.bootstrap import sample_bootstrap
from models.gmm import sample_gmm
from metrics import evaluate_all, evaluate_abl, stratified_subsample, print_dataset_summary
from tables import (
    make_table_full, make_table_sample_size,
    make_table_forward, make_table_reverse,
    make_table_drop_one, save_all_tables,
)

# ── Configuration — only edit this cell to change what gets run ───────────────

# Datasets: add or remove loaders freely. The pipeline adapts to each one.
DATASETS = [load_HIV, load_breast, load_diabetes]

# Sample size as a fraction of each class (scales per dataset automatically).
# e.g. 0.4 on a dataset with 100 class-0 samples → n0=40.
FRACTIONS = [0.2, 0.3, 0.4, 0.5]

# Fraction used for the ablation experiments (forward / reverse / drop-one).
ABLATION_FRACTION = 0.4

# k values for forward and reverse ablation.
# Values larger than a dataset's p are skipped automatically.
FORWARD_KS = [3, 5, 10, 15, 20]
REVERSE_KS  = [1, 3, 5, 10]

CVAE_EPOCHS = 200
SEED        = 42
OUTDIR      = Path('../results')
OUTDIR.mkdir(parents=True, exist_ok=True)

print('Configuration set. Output directory:', OUTDIR)

Configuration set. Output directory: ..\results


---
## 2. Dataset Summary

Check actual class sizes before running — this is what drives all (n0, n1) choices.

In [2]:
print_dataset_summary(DATASETS)

Dataset                  p    Class 0    Class 1    Total
--------------------------------------------------------
HIV                     63         23         68       91
breast_cancer           30        212        357      569
diabetes                 8        500        268      768


c:\Users\tonyt\Desktop\synthetic-data\.venv\Lib\site-packages\sklearn\preprocessing\_label.py:120: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


---
## 3. Helper Functions

In [3]:
def fraction_to_n(y, frac):
    """
    Convert a class fraction to (n0, n1) based on actual class counts.
    Ensures at least 2 samples per class so downstream metrics do not break.
    """
    n0 = max(2, int(np.floor((y == 0).sum() * frac)))
    n1 = max(2, int(np.floor((y == 1).sum() * frac)))
    return n0, n1


def select_feature_subset(X, feature_names, mode='full', ranked_idx=None, k=None, drop_idx=None):
    p = X.shape[1]
    if mode == 'full':
        return np.arange(p)
    elif mode == 'drop_one':
        return np.array([j for j in range(p) if j != drop_idx])
    elif mode == 'forward':
        return np.array(ranked_idx[:k])
    elif mode == 'reverse':
        return np.array(ranked_idx[k:])
    else:
        raise ValueError(f'Unknown mode: {mode}')


def rank_features_by_rf_importance(X, y, seed=42, n_estimators=15):
    rf = RandomForestClassifier(n_estimators=n_estimators, random_state=seed)
    rf.fit(X, y)
    return np.argsort(rf.feature_importances_)[::-1]


def make_subset_metadata(feature_mode, p_total, keep, k=None, drop_idx=None):
    n_used = len(keep)
    if feature_mode == 'full':
        return {'subset_family': 'full',     'subset_label': 'all_features',        'subset_param': np.nan, 'n_features_used': n_used}
    if feature_mode == 'drop_one':
        return {'subset_family': 'drop_one', 'subset_label': f'drop_idx_{drop_idx}', 'subset_param': drop_idx, 'n_features_used': n_used}
    if feature_mode == 'forward':
        return {'subset_family': 'forward',  'subset_label': f'top_{k}',             'subset_param': k,        'n_features_used': n_used}
    if feature_mode == 'reverse':
        return {'subset_family': 'reverse',  'subset_label': f'drop_top_{k}',        'subset_param': k,        'n_features_used': n_used}
    raise ValueError(f'Unknown feature_mode: {feature_mode}')


print('Helpers defined.')

Helpers defined.


---
## 4. Experiment Runner

In [ ]:
def run_experiment(
    datasets,
    feature_mode='full',
    frac=None,
    n0=None, n1=None,
    k=None,
    drop_idx=None,
    rows=None,
    figs=None,
    cvae_epochs=200,
    seed=42,
):
    """
    Universal experiment runner. Works on any dataset in DATASETS.

    Supply either `frac` (fraction of each class, scales per dataset automatically)
    or explicit `n0`/`n1`. If both are given, explicit values win.

    Invalid k or drop_idx values for a given dataset are skipped silently,
    so you can loop over the full range and let the runner filter per dataset.
    """
    if rows is None:
        rows = []

    methods = ['bootstrap', 'gmm', 'cvae']

    for load_fn in datasets:
        data          = load_fn()
        X             = data['X']
        y             = data['y']
        feature_names = data['feature_names']
        p             = X.shape[1]
        dataset_name  = data['dataset']

        # resolve (n0, n1)
        if n0 is None or n1 is None:
            if frac is None:
                raise ValueError('Provide either frac or both n0 and n1.')
            _n0, _n1 = fraction_to_n(y, frac)
        else:
            _n0, _n1 = n0, n1

        # guardrails — skip silently rather than crash
        if feature_mode == 'drop_one' and (drop_idx is None or not (0 <= drop_idx < p)):
            continue
        if feature_mode == 'forward' and (k is None or not (0 < k <= p)):
            print(f'  [{dataset_name}] Skipping forward k={k} (p={p})')
            continue
        if feature_mode == 'reverse' and (k is None or not (0 <= k < p)):
            print(f'  [{dataset_name}] Skipping reverse k={k} (p={p})')
            continue

        X_small, y_small, _ = stratified_subsample(X, y, n0=_n0, n1=_n1, seed=seed)

        ranked_idx        = rank_features_by_rf_importance(X_small, y_small, seed=seed)
        keep              = select_feature_subset(X_small, feature_names, mode=feature_mode,
                                                  ranked_idx=ranked_idx, k=k, drop_idx=drop_idx)
        X_use             = X_small[:, keep]
        feature_names_use = [feature_names[j] for j in keep]
        subset_meta       = make_subset_metadata(feature_mode, p, keep, k=k, drop_idx=drop_idx)

        for method in methods:
            if method == 'bootstrap':
                X_syn, y_syn = sample_bootstrap(X_use, y_small, _n0, _n1, seed=seed)

            elif method == 'gmm':
                X_syn, y_syn = sample_gmm(X_use, y_small, _n0, _n1, seed=seed)

            elif method == 'cvae':
                print(f'  Training CVAE — {dataset_name} | {subset_meta["subset_label"]}')
                best = train_cvae_on_arrays(
                    X_use, y_small, seed=seed,
                    epochs=cvae_epochs, batch_size=32,
                )
                X_syn, y_syn = sample_cvae_dataset(best, _n0, _n1, seed=seed)

            print(f'  Evaluating {dataset_name} | {method} | {subset_meta["subset_label"]} | n=({_n0},{_n1})')
            metrics, fig = evaluate_all(
                X_real=X_use, y_real=y_small,
                X_syn=X_syn,  y_syn=y_syn,
                feature_names=feature_names_use, 
                method=method,  
                frac=frac,     
            )

            row = {
                'dataset':            dataset_name,
                'category':           data['category'],
                'method':             method,
                'n0':                 _n0,
                'n1':                 _n1,
                'frac':               frac,
                'seed':               seed,
                'feature_mode':       feature_mode,
                'subset_family':      subset_meta['subset_family'],
                'subset_label':       subset_meta['subset_label'],
                'subset_param':       subset_meta['subset_param'],
                'n_features_used':    subset_meta['n_features_used'],
                'n_features_total':   p,
                'rf_sep_mean':        metrics.get('rf_sep_mean',        np.nan),
                'rf_sep_sd':          metrics.get('rf_sep_sd',          np.nan),
                'rf_auc_mean':        metrics.get('rf_auc_mean',        np.nan),
                'rf_auc_sd':          metrics.get('rf_auc_sd',          np.nan),
                'disc_f1_mean':       metrics.get('disc_f1_mean',       np.nan),
                'disc_f1_sd':         metrics.get('disc_f1_sd',         np.nan),
                'tstr_f1':            metrics.get('tstr_f1',            np.nan),
                'trtr_f1':            metrics.get('trtr_f1',            np.nan),
                'utility_gap':        metrics.get('utility_gap',        np.nan),
                'kld_mean':           metrics.get('kld_mean',           np.nan),
                'kld_max':            metrics.get('kld_max',            np.nan),
                'corr_mean_abs_diff': metrics.get('corr_mean_abs_diff', np.nan),
                'corr_max_abs_diff':  metrics.get('corr_max_abs_diff',  np.nan),
                'prop_significant':   metrics.get('prop_significant',   np.nan),
                'kept_feature_idx':   list(map(int, keep)),
                'kept_feature_names': feature_names_use,
            }
            rows.append(row)

            if figs is not None:
                figs.append({
                    "dataset":      dataset_name,
                    "method":       method,
                    "feature_mode": feature_mode,
                    "subset_label": subset_meta["subset_label"],
                    "frac":         frac,
                    "n0":           _n0,
                    "n1":           _n1,
                    # raw arrays needed for PCA grid
                    "X_real":       X_use.copy(),
                    "y_real":       y_small.copy(),
                    "X_syn":        X_syn.copy(),
                    "y_syn":        y_syn.copy(),
                    "fig":          fig,
                })
 

            # if isinstance(fig, dict):
            #     for f in fig.values():
            #         if hasattr(f, 'get_axes'): 
            #             plt.close(f)
            # else:
            #     plt.close(fig)

    return rows, figs

print('Runner defined.')

Runner defined.


---
## 5. Run All Experiments

Four blocks, same runner, same datasets every time.
Results accumulate in `all_rows` / `all_figs` then consolidate into `df_all`.

In [5]:
all_rows = []
all_figs = []

# ── Exp 1: Full features — sample size sweep ──────────────────────────────────
# Does quality degrade with less real training data?
print('=' * 60)
print('EXP 1 — Full features, sample size sweep')
print('=' * 60)
for frac in FRACTIONS:
    print(f'\n── frac={frac}')
    run_experiment(
        datasets=DATASETS, feature_mode='full',
        frac=frac,
        rows=all_rows, figs=all_figs,
        cvae_epochs=CVAE_EPOCHS, seed=SEED,
    )

# ── Exp 2: Forward ablation — keep top k features ─────────────────────────────
# Does quality worsen as we include more features (higher dependency complexity)?
print('\n' + '=' * 60)
print('EXP 2 — Forward ablation (keep top k features)')
print('=' * 60)
for k in FORWARD_KS:
    print(f'\n── forward k={k}')
    run_experiment(
        datasets=DATASETS, feature_mode='forward',
        frac=ABLATION_FRACTION, k=k,
        rows=all_rows, figs=all_figs,
        cvae_epochs=CVAE_EPOCHS, seed=SEED,
    )

# ── Exp 3: Reverse ablation — drop top k features ─────────────────────────────
# Is failure concentrated in a few high-importance features, or distributed?
print('\n' + '=' * 60)
print('EXP 3 — Reverse ablation (drop top k features)')
print('=' * 60)
for k in REVERSE_KS:
    print(f'\n── reverse k={k}')
    run_experiment(
        datasets=DATASETS, feature_mode='reverse',
        frac=ABLATION_FRACTION, k=k,
        rows=all_rows, figs=all_figs,
        cvae_epochs=CVAE_EPOCHS, seed=SEED,
    )

# ── Exp 4: Drop-one sensitivity ───────────────────────────────────────────────
# Which single feature removal most changes synthetic quality?
# Loops up to max(p) across all datasets; invalid indices skipped per dataset.
max_p = max(load_fn()['X'].shape[1] for load_fn in DATASETS)
print('\n' + '=' * 60)
print(f'EXP 4 — Drop-one sensitivity (max p={max_p})')
print('=' * 60)
for j in range(max_p):
    run_experiment(
        datasets=DATASETS, feature_mode='drop_one',
        frac=ABLATION_FRACTION, drop_idx=j,
        rows=all_rows, figs=all_figs,
        cvae_epochs=CVAE_EPOCHS, seed=SEED,
    )

# ── Consolidate ───────────────────────────────────────────────────────────────
df_all = pd.DataFrame(all_rows)
print(f'\nTotal rows: {len(df_all)}')
display(
    df_all.groupby(['dataset', 'feature_mode'])
          .size().rename('n_rows').reset_index()
)

EXP 1 — Full features, sample size sweep

── frac=0.2
  Evaluating HIV | bootstrap | all_features | n=(4,13)
  RF trial 1/10


c:\Users\tonyt\Desktop\synthetic-data\.venv\Lib\site-packages\sklearn\preprocessing\_label.py:120: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


  RF trial 10/10
  Evaluating HIV | gmm | all_features | n=(4,13)
  RF trial 1/10
  RF trial 10/10
  Training CVAE — HIV | all_features
Epoch  50 | val loss=66.1914 recon=65.3207 kl=1.7414
Epoch 100 | val loss=63.1995 recon=61.4060 kl=3.5870
Epoch 150 | val loss=60.6539 recon=58.3951 kl=4.5176
Epoch 200 | val loss=61.0844 recon=58.4068 kl=5.3551
  Evaluating HIV | cvae | all_features | n=(4,13)
  RF trial 1/10
  RF trial 10/10
  Evaluating breast_cancer | bootstrap | all_features | n=(42,71)
  RF trial 1/10
  RF trial 10/10
  Evaluating breast_cancer | gmm | all_features | n=(42,71)
  RF trial 1/10
  RF trial 10/10
  Training CVAE — breast_cancer | all_features
Epoch  50 | val loss=16.2833 recon=13.1330 kl=6.3004
Epoch 100 | val loss=14.6000 recon=11.4673 kl=6.2656
Epoch 150 | val loss=13.6612 recon=10.3848 kl=6.5529
Epoch 200 | val loss=12.5720 recon=9.1401 kl=6.8637
  Evaluating breast_cancer | cvae | all_features | n=(42,71)
  RF trial 1/10
  RF trial 10/10
  Evaluating diabetes | b

C:\Users\tonyt\Desktop\synthetic-data\plots.py:146: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, ax = plt.subplots(figsize=(max(8, len(order) * 0.5), 4), constrained_layout=True)


  Evaluating diabetes | gmm | all_features | n=(100,53)
  RF trial 1/10
  RF trial 10/10
  Training CVAE — diabetes | all_features
Epoch  50 | val loss=7.0651 recon=5.1171 kl=3.8961
Epoch 100 | val loss=5.6726 recon=3.5415 kl=4.2621
Epoch 150 | val loss=5.3429 recon=3.1910 kl=4.3038
Epoch 200 | val loss=5.8157 recon=3.6290 kl=4.3734
  Evaluating diabetes | cvae | all_features | n=(100,53)
  RF trial 1/10
  RF trial 10/10

── frac=0.3
  Evaluating HIV | bootstrap | all_features | n=(6,20)
  RF trial 1/10
  RF trial 10/10


...

c:\Users\tonyt\Desktop\synthetic-data\.venv\Lib\site-packages\sklearn\preprocessing\_label.py:120: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


  Evaluating HIV | bootstrap | drop_idx_61 | n=(9,27)
  RF trial 1/10
  RF trial 10/10
  Evaluating HIV | gmm | drop_idx_61 | n=(9,27)
  RF trial 1/10
  RF trial 10/10
  Training CVAE — HIV | drop_idx_61
Epoch  50 | val loss=649.3948 recon=645.8100 kl=7.1696
Epoch 100 | val loss=641.3311 recon=633.2421 kl=16.1782
Epoch 150 | val loss=616.5181 recon=605.6886 kl=21.6590
Epoch 200 | val loss=602.5164 recon=587.6071 kl=29.8185
  Evaluating HIV | cvae | drop_idx_61 | n=(9,27)
  RF trial 1/10
  RF trial 10/10


c:\Users\tonyt\Desktop\synthetic-data\.venv\Lib\site-packages\sklearn\preprocessing\_label.py:120: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


  Evaluating HIV | bootstrap | drop_idx_62 | n=(9,27)
  RF trial 1/10
  RF trial 10/10
  Evaluating HIV | gmm | drop_idx_62 | n=(9,27)
  RF trial 1/10
  RF trial 10/10
  Training CVAE — HIV | drop_idx_62
Epoch  50 | val loss=650.1691 recon=646.4598 kl=7.4186
Epoch 100 | val loss=643.2623 recon=634.8849 kl=16.7548
Epoch 150 | val loss=616.3623 recon=605.5900 kl=21.5447
Epoch 200 | val loss=598.0804 recon=584.3127 kl=27.5355
  Evaluating HIV | cvae | drop_idx_62 | n=(9,27)
  RF trial 1/10
  RF trial 10/10

Total rows: 408


,dataset,feature_mode,n_rows
0,HIV,drop_one,189
1,HIV,forward,15
2,HIV,full,12
3,HIV,reverse,12
4,breast_cancer,drop_one,90
5,breast_cancer,forward,15
6,breast_cancer,full,12
7,breast_cancer,reverse,12
8,diabetes,drop_one,24
9,diabetes,forward,6


In [6]:
df_all.to_csv(OUTDIR / 'synthetic_results_all.csv', index=False)
with open(OUTDIR / 'all_figs.pkl', 'wb') as f:
    pickle.dump(all_figs, f)
print('Saved to', OUTDIR)

Saved to ..\results
